In [1]:
%load_ext autoreload
%autoreload 2

In [15]:
from package.flows.online.flow import get_online_flow, Shared
from uuid import uuid4
from broflow import state

In [16]:
def generate_experiments(
    embed_methods=['raw'],
    chat_system_prompt_paths=['./package/flows/online/prompts/v1/chat.md'],
    term_detector_system_prompt_paths=["./package/flows/online/prompts/term_detector.md"],
    chat_models=["us.meta.llama3-2-11b-instruct-v1:0"],
    term_detector_models=["us.meta.llama3-2-11b-instruct-v1:0"]
):
    from itertools import product
    
    is_terms = ['skip', 'evidence', 'explanation']
    is_contexts = [True, False]
    
    experiments = []
    for (embed_method, chat_prompt, term_prompt, chat_model, term_model, 
         is_term, is_context) in product(
        embed_methods, chat_system_prompt_paths, term_detector_system_prompt_paths,
        chat_models, term_detector_models, is_terms, is_contexts
    ):
        base_config = {
            "embed_method": embed_method,
            "chat_system_prompt_path": chat_prompt,
            "term_detector_system_prompt_path": term_prompt,
            "chat_model": chat_model,
            "term_detector_model": term_model,
            "is_term": is_term,
            "is_context": is_context
        }
        
        if is_context:
            experiments.extend([
                {**base_config, "is_rerank": False},
                {**base_config, "is_rerank": True}
            ])
        else:
            experiments.append({**base_config, "is_rerank": False})
    
    return experiments


In [17]:
from dataclasses import dataclass, asdict

@dataclass
class ControlExperiment:
    chat_model:str
    chat_system_prompt_path:str
    term_detector_model:str
    term_detector_system_prompt_path:str
    embed_method:str
    is_term:str
    is_context:bool
    is_rerank:bool

In [18]:
experiments = generate_experiments(
    embed_methods=['raw'],
    chat_system_prompt_paths=['./package/flows/online/prompts/v1/chat.md'],
    term_detector_system_prompt_paths=["./package/flows/online/prompts/term_detector.md"],
    chat_models=["us.meta.llama3-2-11b-instruct-v1:0"],
    term_detector_models=["us.meta.llama3-2-11b-instruct-v1:0"]
)

In [19]:
import json

with open("./dataset/trainset.json", 'r', encoding='utf-8') as f:
    trainset = json.load(f)

In [20]:
for ts in trainset:
    ts['metadata']['source'] = ts['metadata']['source'].replace(":", "")

In [21]:
def one_experiment(ts, con_exp:ControlExperiment):
    _id = str(uuid4())
    experiment_storage = "./evaluations/{file}.json".format(file=_id)
    experiment = _id
    shared = Shared(
        experiment=experiment,
        question=ts['question'],
        answer=ts['answer'],
        type=ts['type'],
        source=ts['metadata']['source'],
        chat_model=con_exp.chat_model,
        term_detector_model=con_exp.term_detector_model,
        chat_system_prompt_path=con_exp.chat_system_prompt_path,
        term_detector_system_prompt_path=con_exp.term_detector_system_prompt_path,
        experiment_storage=experiment_storage,
        embed_method=con_exp.embed_method,
        is_term=con_exp.is_term,
        is_context=con_exp.is_context,
        is_rerank=con_exp.is_rerank

    )

    flow = get_online_flow(experiment=shared.experiment)
    flow.run(shared)    

def main(trainset, experiments):
    for experiment in experiments:
        con_exp = ControlExperiment(**experiment)
        for ts in trainset:
            one_experiment(ts, con_exp)

In [22]:
state.set('debug', False)
main(trainset[0:5], experiments[0:1])

Start 04b4b2aa-8773-4319-bf83-feb9e5757838


d:\broai-arai\backend\.venv\Lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


End 04b4b2aa-8773-4319-bf83-feb9e5757838
Start 6254e9d6-d607-48aa-b0d5-c46b3b6ea703
End 6254e9d6-d607-48aa-b0d5-c46b3b6ea703
Start 93651ac7-80f7-4a2b-bcb6-be37ce98c52f
End 93651ac7-80f7-4a2b-bcb6-be37ce98c52f
Start 9f599a69-985a-4403-8d22-a528509cff6d
End 9f599a69-985a-4403-8d22-a528509cff6d
Start af6a3442-1255-425d-ab1c-129d082bf318
End af6a3442-1255-425d-ab1c-129d082bf318
